# Task 4: TF-IDF Vectorization and Similarity Analysis

This notebook converts cleaned TED Talk text into TF-IDF vectors and uses cosine similarity to find five talks with text most similar to a selected talk. The cleaned CSV contains separate text fields rather than a `combined_text` column, so the column is assembled below from the available cleaned fields.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity



df1 = pd.read_csv("C:/Users/ajayr/Desktop/TED-TALK-RECOMMENDATION-SYSTEM/data/processed/ted_talks_cleaned.csv")

df1.head()

,comments,description,duration,event,film_date,languages,main_speaker,name,num_speaker,published_date,ratings,related_talks,speaker_occupation,tags,title,url,views,transcript
0,459.875,sir ken robinson makes an entertaining and pro...,1164.0,TED2006,1140825600,48,Ken Robinson,Ken Robinson: Do schools kill creativity?,1,1151367060,"[{'id': 7, 'name': 'Funny', 'count': 19645}, {...","[{'id': 865, 'hero': 'https://pe.tedcdn.com/im...",Author/educator,"['children', 'creativity', 'culture', 'dance',...",do schools kill creativity?,https://www.ted.com/talks/ken_robinson_says_sc...,3118209.625,good morning. how are you?(laughter)it's been ...
1,265.000,with the same humor and humanity he exuded in ...,977.0,TED2006,1140825600,43,Al Gore,Al Gore: Averting the climate crisis,1,1151367060,"[{'id': 7, 'name': 'Funny', 'count': 544}, {'i...","[{'id': 243, 'hero': 'https://pe.tedcdn.com/im...",Climate advocate,"['alternative energy', 'cars', 'climate change...",averting the climate crisis,https://www.ted.com/talks/al_gore_on_averting_...,3118209.625,"thank you so much, chris. and it's truly a gre..."
2,124.000,new york times columnist david pogue takes aim...,1286.0,TED2006,1140739200,26,David Pogue,David Pogue: Simplicity sells,1,1151367060,"[{'id': 7, 'name': 'Funny', 'count': 964}, {'i...","[{'id': 1725, 'hero': 'https://pe.tedcdn.com/i...",Technology columnist,"['computers', 'entertainment', 'interface desi...",simplicity sells,https://www.ted.com/talks/david_pogue_says_sim...,1636292.000,"(music: ""the sound of silence,"" simon & garfun..."
3,200.000,"in an emotionally charged talk, macarthur-winn...",1116.0,TED2006,1140912000,35,Majora Carter,Majora Carter: Greening the ghetto,1,1151367060,"[{'id': 3, 'name': 'Courageous', 'count': 760}...","[{'id': 1041, 'hero': 'https://pe.tedcdn.com/i...",Activist for environmental justice,"['macarthur grant', 'activism', 'business', 'c...",greening the ghetto,https://www.ted.com/talks/majora_carter_s_tale...,1697550.000,if you're here today — and i'm very happy that...
4,459.875,you've never seen data presented like this. wi...,1190.0,TED2006,1140566400,48,Hans Rosling,Hans Rosling: The best stats you've ever seen,1,1151440680,"[{'id': 9, 'name': 'Ingenious', 'count': 3202}...","[{'id': 2056, 'hero': 'https://pe.tedcdn.com/i...",Global health expert; data visionary,"['africa', 'asia', 'google', 'demo', 'economic...",the best stats you've ever seen,https://www.ted.com/talks/hans_rosling_shows_t...,3118209.625,"about 10 years ago, i took on the task to teac..."


In [2]:
df1.shape

(2458, 18)

In [3]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 2458 entries, 0 to 2457
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   comments            2458 non-null   float64
 1   description         2458 non-null   str    
 2   duration            2458 non-null   float64
 3   event               2458 non-null   str    
 4   film_date           2458 non-null   int64  
 5   languages           2458 non-null   int64  
 6   main_speaker        2458 non-null   str    
 7   name                2458 non-null   str    
 8   num_speaker         2458 non-null   int64  
 9   published_date      2458 non-null   int64  
 10  ratings             2458 non-null   str    
 11  related_talks       2458 non-null   str    
 12  speaker_occupation  2458 non-null   str    
 13  tags                2458 non-null   str    
 14  title               2458 non-null   str    
 15  url                 2458 non-null   str    
 16  views            

In [4]:
# df.iloc[0]["related_talks"]

In [6]:
# columns to keep
df=df1[["url",'title','transcript','ratings','tags','main_speaker','description','speaker_occupation']]
df.head()

,url,title,transcript,ratings,tags,main_speaker,description,speaker_occupation
0,https://www.ted.com/talks/ken_robinson_says_sc...,do schools kill creativity?,good morning. how are you?(laughter)it's been ...,"[{'id': 7, 'name': 'Funny', 'count': 19645}, {...","['children', 'creativity', 'culture', 'dance',...",Ken Robinson,sir ken robinson makes an entertaining and pro...,Author/educator
1,https://www.ted.com/talks/al_gore_on_averting_...,averting the climate crisis,"thank you so much, chris. and it's truly a gre...","[{'id': 7, 'name': 'Funny', 'count': 544}, {'i...","['alternative energy', 'cars', 'climate change...",Al Gore,with the same humor and humanity he exuded in ...,Climate advocate
2,https://www.ted.com/talks/david_pogue_says_sim...,simplicity sells,"(music: ""the sound of silence,"" simon & garfun...","[{'id': 7, 'name': 'Funny', 'count': 964}, {'i...","['computers', 'entertainment', 'interface desi...",David Pogue,new york times columnist david pogue takes aim...,Technology columnist
3,https://www.ted.com/talks/majora_carter_s_tale...,greening the ghetto,if you're here today — and i'm very happy that...,"[{'id': 3, 'name': 'Courageous', 'count': 760}...","['macarthur grant', 'activism', 'business', 'c...",Majora Carter,"in an emotionally charged talk, macarthur-winn...",Activist for environmental justice
4,https://www.ted.com/talks/hans_rosling_shows_t...,the best stats you've ever seen,"about 10 years ago, i took on the task to teac...","[{'id': 9, 'name': 'Ingenious', 'count': 3202}...","['africa', 'asia', 'google', 'demo', 'economic...",Hans Rosling,you've never seen data presented like this. wi...,Global health expert; data visionary


In [7]:
df.duplicated(subset='url').sum()

np.int64(0)

In [8]:
print(repr(df['tags'].iloc[0]))
print(repr(df['ratings'].iloc[0]))
print(repr(df['speaker_occupation'].iloc[0]))

"['children', 'creativity', 'culture', 'dance', 'education', 'parenting', 'teaching']"
"[{'id': 7, 'name': 'Funny', 'count': 19645}, {'id': 1, 'name': 'Beautiful', 'count': 4573}, {'id': 9, 'name': 'Ingenious', 'count': 6073}, {'id': 3, 'name': 'Courageous', 'count': 3253}, {'id': 11, 'name': 'Longwinded', 'count': 387}, {'id': 2, 'name': 'Confusing', 'count': 242}, {'id': 8, 'name': 'Informative', 'count': 7346}, {'id': 22, 'name': 'Fascinating', 'count': 10581}, {'id': 21, 'name': 'Unconvincing', 'count': 300}, {'id': 24, 'name': 'Persuasive', 'count': 10704}, {'id': 23, 'name': 'Jaw-dropping', 'count': 4439}, {'id': 25, 'name': 'OK', 'count': 1174}, {'id': 26, 'name': 'Obnoxious', 'count': 209}, {'id': 10, 'name': 'Inspiring', 'count': 24924}]"
'Author/educator'


In [9]:
import ast

cols = ['title', 'url', 'main_speaker', 'speaker_occupation',
        'description', 'tags', 'ratings', 'transcript']
data = df[cols].copy()   # work on a copy, so df stays untouched

def top_ratings(x, top_n=3):
    items = ast.literal_eval(x)
    items = sorted(items, key=lambda d: d['count'], reverse=True)
    return [d['name'] for d in items[:top_n]]

# parse from df (not data), so re-running this cell is safe
data['tags'] = df['tags'].apply(ast.literal_eval)
data['ratings'] = df['ratings'].apply(top_ratings)

print(type(data['tags'].iloc[0]), data['tags'].iloc[0])
print(type(data['ratings'].iloc[0]), data['ratings'].iloc[0])
print(data['tags'].apply(len).describe())

<class 'list'> ['children', 'creativity', 'culture', 'dance', 'education', 'parenting', 'teaching']
<class 'list'> ['Inspiring', 'Funny', 'Persuasive']
count    2458.000000
mean        7.564280
std         4.334242
min         1.000000
25%         5.000000
50%         6.000000
75%         9.000000
max        32.000000
Name: tags, dtype: float64


In [10]:
import re

def collapse(item):
    # "Ken Robinson" -> "kenrobinson", "climate change" -> "climatechange"
    return re.sub(r'[^a-z0-9]', '', str(item).lower())

# step 4: make lists
data['main_speaker'] = data['main_speaker'].apply(lambda x: [x])
data['speaker_occupation'] = data['speaker_occupation'].apply(
    lambda x: [p.strip().lower() for p in re.split(r'[;/,]', str(x)) if p.strip()]
)

# step 5: remove spaces/punctuation inside items (not for occupation)
for col in ['main_speaker', 'tags', 'ratings']:
    data[col] = data[col].apply(lambda lst: [collapse(i) for i in lst if collapse(i)])

print(data.loc[0, ['main_speaker', 'speaker_occupation', 'tags', 'ratings']].to_dict())
print(data['speaker_occupation'].apply(len).value_counts().head())
print(data['main_speaker'].apply(len).value_counts())
print(data['tags'].explode().str.len().describe())
print(data['tags'].explode().sample(10, random_state=1).tolist())

{'main_speaker': ['kenrobinson'], 'speaker_occupation': ['author', 'educator'], 'tags': ['children', 'creativity', 'culture', 'dance', 'education', 'parenting', 'teaching'], 'ratings': ['inspiring', 'funny', 'persuasive']}
speaker_occupation
1    2078
2     321
3      55
4       3
5       1
Name: count, dtype: int64
main_speaker
1    2458
Name: count, dtype: int64
count    18593.000000
mean         8.486151
std          3.147844
min          2.000000
25%          6.000000
50%          8.000000
75%         11.000000
max         22.000000
Name: tags, dtype: float64
['animals', 'globalissues', 'plastic', 'women', 'globalissues', 'life', 'technology', 'mars', 'design', 'innovation']


In [11]:
# Next is step 6, cleaning the free text in description and transcript

import re, html, time
import pandas as pd
from collections import Counter
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# ---------- settings ----------
KEEP  = {'system', 'fire', 'interest'}   # real content words that sklearn lists as stop words
EXTRA = {'ive','im','weve','dont','youre','thats','theyre','youve','theyve','youll',
         'didnt','doesnt','isnt','wasnt','cant','wont','couldnt','wouldnt','shouldnt',
         'hasnt','havent','arent','werent','hes','shes','lets','theres','whats','heres','just'}

ps = PorterStemmer()
base_stop = (set(ENGLISH_STOP_WORDS) - KEEP) | EXTRA
leak = (base_stop | {ps.stem(w) for w in base_stop}) - KEEP   # stop words + their stemmed forms

ann = re.compile(r'\[(unclear|bleep|music|applause|laughter|video|audio)[^\]]*\]', re.I)

def clean_text(text):
    text = html.unescape(str(text))
    text = ann.sub(' ', text)                  # [unclear], [bleep]...
    text = re.sub(r'<[^>]+>', ' ', text)       # html tags
    text = re.sub(r'\([^)]*\)', ' ', text)     # (laughter), (applause)...
    text = text.lower()
    text = re.sub(r"[’'`]", '', text)          # don't -> dont
    text = re.sub(r'[^a-z\s]', ' ', text)      # letters only
    return re.sub(r'\s+', ' ', text).strip()

cache = {}
def stem_text(text):
    out = []
    for w in text.split():
        if w in leak:                          # stop word before stemming
            continue
        s = cache.get(w)
        if s is None:
            s = cache[w] = ps.stem(w)
        if s not in leak:                      # stop word created by stemming (going -> go)
            out.append(s)
    return ' '.join(out)

# ---------- text pieces (always rebuilt from df) ----------
t0 = time.time()
title_txt = df['title'].apply(lambda t: stem_text(clean_text(t)))
desc_txt  = df['description'].apply(lambda t: stem_text(clean_text(t)))
trans_txt = df['transcript'].apply(lambda t: stem_text(clean_text(t)))
occ_txt   = data['speaker_occupation'].apply(lambda l: stem_text(clean_text(' '.join(l))))
print(f'time: {time.time()-t0:.1f}s')
data['description'], data['transcript'] = desc_txt, trans_txt

# ---------- steps 8-9: combine ----------
join_list = lambda l: ' '.join(l)
meta = (title_txt + ' ' + desc_txt + ' ' + data['tags'].apply(join_list) + ' ' +
        data['main_speaker'].apply(join_list) + ' ' + occ_txt + ' ' +
        data['ratings'].apply(join_list))
meta = meta.str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()
content = (meta + ' ' + trans_txt).str.replace(r'\s+', ' ', regex=True).str.strip()

new_df = pd.DataFrame({
    'id': range(len(df)),
    'title': df['title'].values,
    'url': df['url'].str.strip().values,
    'main_speaker': df['main_speaker'].apply(lambda x: x[0] if isinstance(x, list) else x).values,
    'meta': meta.values,
    'tags': content.values,
})

# ---------- verify ----------
free = Counter(' '.join(desc_txt + ' ' + trans_txt).split())
print('leaked after:', [w for w in free if w in leak])
print('kept words  :', {w: free.get(w, 0) for w in sorted(KEEP)})
print('shape:', new_df.shape, '| nulls:', int(new_df.isnull().sum().sum()),
      '| empty transcripts:', int((trans_txt.str.len() == 0).sum()),
      '| talks < 20 words:', int((new_df['tags'].str.split().str.len() < 20).sum()))
print('\nwords per talk:'); print(new_df['tags'].str.split().str.len().describe().round(1))
doc_freq = Counter(w for t in (desc_txt + ' ' + trans_txt) for w in set(t.split()))
print('\nmost widespread words (% of talks):',
      [(w, f'{c/len(df):.0%}') for w, c in doc_freq.most_common(30)])
print('\nmost frequent tokens:', Counter(' '.join(new_df['tags']).split()).most_common(20))
print('\nMETA[0]:', new_df.loc[0, 'meta'][:300])

time: 13.8s
leaked after: []
kept words  : {'fire': 691, 'interest': 3147, 'system': 4014}
shape: (2458, 6) | nulls: 0 | empty transcripts: 8 | talks < 20 words: 0

words per talk:
count    2458.0
mean      860.5
std       371.7
min        29.0
25%       591.0
50%       873.5
75%      1112.8
max      3728.0
Name: tags, dtype: float64

most widespread words (% of talks): [('like', '96%'), ('thank', '93%'), ('time', '92%'), ('year', '90%'), ('make', '90%'), ('know', '89%'), ('peopl', '89%'), ('way', '89%'), ('thing', '88%'), ('look', '88%'), ('think', '88%'), ('want', '86%'), ('work', '85%'), ('use', '84%'), ('come', '84%'), ('say', '83%'), ('world', '83%'), ('realli', '82%'), ('right', '80%'), ('start', '79%'), ('talk', '79%'), ('need', '76%'), ('new', '76%'), ('actual', '75%'), ('day', '74%'), ('live', '74%'), ('differ', '74%'), ('happen', '74%'), ('lot', '72%'), ('did', '72%')]

most frequent tokens: [('like', 20456), ('peopl', 20079), ('thing', 14964), ('think', 14655), ('know', 1409

## TF-IDF, similarity and recommendation test

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

tfidf = TfidfVectorizer(max_features=10000, min_df=2, max_df=0.7, sublinear_tf=True)
tfidf_matrix = tfidf.fit_transform(new_df['tags'])
terms = tfidf.get_feature_names_out()

print('matrix shape:', tfidf_matrix.shape)
print('non-zero terms per talk (median):', int(pd.Series(tfidf_matrix.getnnz(axis=1)).median()))
print('highest total weight:', terms[tfidf_matrix.sum(axis=0).A1.argsort()[::-1][:20]].tolist())
print('lowest idf (most common kept):', terms[tfidf.idf_.argsort()[:15]].tolist())

# TF-IDF rows are L2-normalised, so linear_kernel == cosine similarity
sim = linear_kernel(tfidf_matrix)
print('\nsimilarity shape:', sim.shape, '| diagonal all 1:', bool((sim.diagonal().round(6) == 1).all()))

def recommend(title, n=5):
    idx = new_df.index[new_df['title'].str.lower() == title.lower()][0]
    top = sim[idx].argsort()[::-1][1:n+1]
    out = new_df.loc[top, ['title', 'main_speaker']].copy()
    out['score'] = sim[idx][top].round(3)
    return out

tests = [new_df.loc[0, 'title']] + new_df['title'].sample(2, random_state=0).tolist()
for t in tests:
    print('\n>>>', t)
    print(recommend(t).to_string(index=False))

matrix shape: (2458, 10000)
non-zero terms per talk (median): 368
highest total weight: ['said', 'chang', 'got', 'kind', 'tri', 'good', 'tell', 'human', 'idea', 'ask', 'creat', 'problem', 'learn', 'stori', 'today', 'place', 'help', 'question', 'feel', 'build']
lowest idf (most common kept): ['good', 'tell', 'tri', 'got', 'inspiring', 'chang', 'kind', 'said', 'ask', 'informative', 'today', 'end', 'place', 'turn', 'idea']

similarity shape: (2458, 2458) | diagonal all 1: True

>>> do schools kill creativity?
                                 title         main_speaker  score
how to escape education's death valley         Ken Robinson  0.234
     bring on the learning revolution!         Ken Robinson  0.226
              be an artist, right now!         Young-ha Kim  0.211
            how a boy became an artist Jarrett J. Krosoczka  0.210
                     letting go of god        Julia Sweeney  0.200

>>> how to make a splash in social media
                                            

# Reading the top 5 only gives a feel for quality. TED’s own related_talks column gives a number to compare settings against, and it lets us pick max_df from data instead of guessing.

In [13]:
import ast, numpy as np

title2idx = {t.lower().strip(): i for i, t in enumerate(new_df['title'])}

def parse_related(x):
    try:
        return [d['title'].lower().strip() for d in ast.literal_eval(x)]
    except Exception:
        return []

related_idx = df1['related_talks'].apply(
    lambda x: [title2idx[t] for t in parse_related(x) if t in title2idx])
print(repr(df1['related_talks'].iloc[0])[:250])
print('related talks found in dataset per talk:', related_idx.apply(len).describe().round(2).to_dict())

def evaluate(sim, k=5):
    s = sim.copy()
    np.fill_diagonal(s, -1)
    top = np.argsort(-s, axis=1)[:, :k]
    hit, overlap = [], []
    for i, rel in enumerate(related_idx):
        if not rel:
            continue
        got = len(set(top[i]) & set(rel))
        hit.append(got > 0)
        overlap.append(got / min(len(rel), k))
    return {'hit@5': round(float(np.mean(hit)), 3),
            'overlap@5': round(float(np.mean(overlap)), 3),
            'talks': len(hit)}

N = len(new_df)
print('\nrandom baseline :', evaluate(np.random.default_rng(0).random((N, N))))
print('meta only       :', evaluate(linear_kernel(
    TfidfVectorizer(min_df=2, max_df=0.7, sublinear_tf=True).fit_transform(new_df['meta']))))
for md in [0.5, 0.7, 0.9]:
    m = TfidfVectorizer(max_features=10000, min_df=2, max_df=md, sublinear_tf=True).fit_transform(new_df['tags'])
    print(f'full, max_df={md}:', evaluate(linear_kernel(m)))

'[{\'id\': 865, \'hero\': \'https://pe.tedcdn.com/images/ted/172559_800x600.jpg\', \'speaker\': \'Ken Robinson\', \'title\': \'Bring on the learning revolution!\', \'duration\': 1008, \'slug\': \'sir_ken_robinson_bring_on_the_revolution\', \'viewed_c
related talks found in dataset per talk: {'count': 2458.0, 'mean': 5.94, 'std': 0.26, 'min': 4.0, '25%': 6.0, '50%': 6.0, '75%': 6.0, 'max': 6.0}

random baseline : {'hit@5': 0.009, 'overlap@5': 0.002, 'talks': 2458}
meta only       : {'hit@5': 0.753, 'overlap@5': 0.268, 'talks': 2458}
full, max_df=0.5: {'hit@5': 0.852, 'overlap@5': 0.343, 'talks': 2458}
full, max_df=0.7: {'hit@5': 0.849, 'overlap@5': 0.341, 'talks': 2458}
full, max_df=0.9: {'hit@5': 0.841, 'overlap@5': 0.335, 'talks': 2458}


# Next is the weighted version I mentioned. It builds separate TF-IDF similarities for the metadata and for the transcript, then mixes them. This fixes the imbalance of about 45 meta words against about 800 transcript words. The cell also tests lower max_df values.


In [14]:
def tfidf_sim(texts, max_df, min_df=2, max_features=None):
    m = TfidfVectorizer(max_features=max_features, min_df=min_df,
                        max_df=max_df, sublinear_tf=True).fit_transform(texts)
    return linear_kernel(m)

# 1) lower max_df on the combined text
for md in [0.2, 0.3, 0.4]:
    print(f'full, max_df={md}:', evaluate(tfidf_sim(new_df['tags'], md, max_features=10000)))

# 2) separate similarities, mixed with a weight
sim_meta  = tfidf_sim(new_df['meta'], 0.5)
sim_trans = tfidf_sim(trans_txt, 0.5, max_features=10000)
print()
for w in [0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 1.0]:
    print(f'meta weight {w}:', evaluate(w * sim_meta + (1 - w) * sim_trans))

full, max_df=0.2: {'hit@5': 0.835, 'overlap@5': 0.338, 'talks': 2458}
full, max_df=0.3: {'hit@5': 0.855, 'overlap@5': 0.344, 'talks': 2458}
full, max_df=0.4: {'hit@5': 0.856, 'overlap@5': 0.344, 'talks': 2458}

meta weight 0.0: {'hit@5': 0.826, 'overlap@5': 0.325, 'talks': 2458}
meta weight 0.2: {'hit@5': 0.897, 'overlap@5': 0.381, 'talks': 2458}
meta weight 0.3: {'hit@5': 0.892, 'overlap@5': 0.382, 'talks': 2458}
meta weight 0.4: {'hit@5': 0.88, 'overlap@5': 0.376, 'talks': 2458}
meta weight 0.5: {'hit@5': 0.865, 'overlap@5': 0.358, 'talks': 2458}
meta weight 0.6: {'hit@5': 0.843, 'overlap@5': 0.339, 'talks': 2458}
meta weight 1.0: {'hit@5': 0.753, 'overlap@5': 0.268, 'talks': 2458}


# The next cell sanity-checks 0.1, 0.15 and 0.25, builds the final similarity matrix, re-runs your three test talks, and saves what the website needs. It also strips the stray quote in Christopher "moot" Poole"

In [15]:
import os

W = 0.25
for w in [0.1, 0.15, 0.25]:
    print(f'meta weight {w}:', evaluate(w * sim_meta + (1 - w) * sim_trans))

final_sim = (W * sim_meta + (1 - W) * sim_trans).astype(np.float32)
print('\nFINAL (W=0.25):', evaluate(final_sim))

# display cleanup: remove a stray unmatched trailing quote
new_df['main_speaker'] = new_df['main_speaker'].apply(
    lambda s: s[:-1] if s.endswith('"') and s.count('"') % 2 == 1 else s)

def recommend(title, n=5):
    idx = new_df.index[new_df['title'].str.lower() == title.lower()][0]
    top = final_sim[idx].argsort()[::-1][1:n+1]
    out = new_df.loc[top, ['title', 'main_speaker']].copy()
    out['score'] = final_sim[idx][top].round(3)
    return out

for t in tests:
    print('\n>>>', t)
    print(recommend(t).to_string(index=False))

# ---- save for the website ----
os.makedirs('../artifacts', exist_ok=True)
K = 10
s = final_sim.copy(); np.fill_diagonal(s, -1)
nbr_idx = np.argsort(-s, axis=1)[:, :K]
nbr_score = np.take_along_axis(s, nbr_idx, axis=1).round(4)

# np.save('../artifacts/neighbors_idx.npy', nbr_idx)
# np.save('../artifacts/neighbors_score.npy', nbr_score)
# np.save('../artifacts/similarity_float32.npy', final_sim)
# new_df[['id', 'title', 'url', 'main_speaker']].to_csv('../artifacts/talks.csv', index=False)
new_df[['id', 'title', 'url', 'main_speaker']]

for f in sorted(os.listdir('../artifacts')):
    print(f, round(os.path.getsize(f'../artifacts/{f}') / 1e6, 2), 'MB')

meta weight 0.1: {'hit@5': 0.873, 'overlap@5': 0.358, 'talks': 2458}
meta weight 0.15: {'hit@5': 0.889, 'overlap@5': 0.371, 'talks': 2458}
meta weight 0.25: {'hit@5': 0.897, 'overlap@5': 0.386, 'talks': 2458}

FINAL (W=0.25): {'hit@5': 0.897, 'overlap@5': 0.386, 'talks': 2458}

>>> do schools kill creativity?
                                    title main_speaker  score
   how to escape education's death valley Ken Robinson  0.264
        bring on the learning revolution! Ken Robinson  0.258
              my wish: once upon a school  Dave Eggers  0.181
my wish: find the next einstein in africa   Neil Turok  0.179
         teach arts and sciences together  Mae Jemison  0.177

>>> how to make a splash in social media
                              title             main_speaker  score
      the case for anonymity online Christopher "moot" Poole  0.141
  how social media can make history              Clay Shirky  0.131
   the web is more than "better tv"          Peter Hirshberg  0.127
   

In [16]:
def make_results(selected_idx, k=5):
    rows = []
    for i in selected_idx:
        s = final_sim[i].copy()
        s[i] = -1                                   # never recommend the talk itself
        for j in np.argsort(-s)[:k]:
            rows.append({'Selected Talk': new_df.at[i, 'title'],
                         'Recommended Talk': new_df.at[j, 'title'],
                         'Similarity Score': round(float(final_sim[i, j]), 4)})
    return pd.DataFrame(rows)

# sample: the 3 test talks + 7 random ones
test_idx = [title2idx[t.lower().strip()] for t in tests]
rng = np.random.default_rng(42)
others = [int(i) for i in rng.choice(len(new_df), 12, replace=False) if i not in test_idx][:7]

results_df = make_results(test_idx + others)
results_df.to_csv('../artifacts/ted_talks_recommendations.csv', index=False)
print('Recommendations saved to: ../artifacts/ted_talks_recommendations.csv')
pd.set_option('display.max_colwidth', 70)
display(results_df)

# full file: every talk with its 5 recommendations
all_df = make_results(range(len(new_df)))
all_df.to_csv('../artifacts/ted_talks_recommendations_all.csv', index=False)
print('all talks:', all_df.shape)

Recommendations saved to: ../artifacts/ted_talks_recommendations.csv


,Selected Talk,Recommended Talk,Similarity Score
0,do schools kill creativity?,how to escape education's death valley,0.2644
1,do schools kill creativity?,bring on the learning revolution!,0.2584
2,do schools kill creativity?,my wish: once upon a school,0.1813
3,do schools kill creativity?,my wish: find the next einstein in africa,0.1794
4,do schools kill creativity?,teach arts and sciences together,0.1769
5,how to make a splash in social media,the case for anonymity online,0.1406
6,how to make a splash in social media,how social media can make history,0.1309
7,how to make a splash in social media,"the web is more than ""better tv""",0.1275
8,how to make a splash in social media,the next web,0.1253
9,how to make a splash in social media,why you should care about whale poo,0.1233


all talks: (12290, 3)


In [18]:
print("\nLowest-scoring recommendations:")
display(all_df.nsmallest(5, "Similarity Score"))

print("\nHighest-scoring recommendations:")
display(all_df.nlargest(5, "Similarity Score"))


Lowest-scoring recommendations:


,Selected Talk,Recommended Talk,Similarity Score
734,my magic moves,"the dancer, the singer, the cellist ... and a moment of creative m...",0.0387
733,my magic moves,a choreographer's creative process in real time,0.0402
732,my magic moves,dance to change the world,0.0403
8949,"a dance in a hurricane of paper, wind and light",dancing with light,0.0480
8948,"a dance in a hurricane of paper, wind and light","the dancer, the singer, the cellist ... and a moment of creative m...",0.0488



Highest-scoring recommendations:


,Selected Talk,Recommended Talk,Similarity Score
6495,teen wonders play bluegrass,bluegrass virtuosity from ... new jersey?,0.7159
7265,bluegrass virtuosity from ... new jersey?,teen wonders play bluegrass,0.7159
2905,growing new organs,printing a human kidney,0.4907
4230,printing a human kidney,growing new organs,0.4907
3190,glowing life in an underwater world,"the weird, wonderful world of bioluminescence",0.4577
